In [1]:

from utils import goto_project_root
from utils.path_settings import MODEL_SAVE_PATH, DATA_PATH, LOG_PATH, CONFIG_PATH
from torch.utils.tensorboard import SummaryWriter
import SimulateDatasets.GenTrainingData as g
from utils import create_splits, get_dataloaders, force_remove_dir
from Network_models import Trainer as t
from importlib import reload
import os
reload(t)
reload(g)
import time
import torch
import numpy as np
from pytorch3d.transforms import so3_relative_angle
from pytorch3d.transforms import quaternion_to_matrix
import json
import Train_task.train_from_config_names as train
reload(train)

<module 'Train_task.train_from_config_names' from 'C:\\Users\\timmy\\Documents\\Projects_dir\\Mental_Rotations\\mental-rotations\\Train_task\\train_from_config_names.py'>

In [4]:
a = torch.tensor([1, 2, 3], dtype=torch.float32).cuda()
print(a.cpu().type())

torch.FloatTensor


In [3]:
task = "0.1q"
network_name_epochs = {
    "FC_16_GRU_1layer_8hidden": 100, 
}
special_names = "constant"
trainer = train.train_from_config_names(network_name_epochs, task, 1, special_names)

Successfully removed directory: D:\Projects\mental-rotations\models\FC_16_GRU_1layer_8hidden_0.1q_model_gradual_constant
Successfully removed directory: D:\Projects\mental-rotations\logs\FC_16_GRU_1layer_8hidden_0.1q_model_gradual_constant
Successfully removed directory: D:\Projects\mental-rotations\models\FC_16_GRU_1layer_8hidden_model_checkpoints_gradual_constant
Data_0.1q.pth
['Data_0.1qimages', 'Data_0.2q.pth', 'Data_0.2q_lean.pth', 'Data_0.2q_long_32_2.pth', 'Data_1.1.pth', 'Data_1.1images', 'saved_images']
Generating training data for FC_16_GRU_1layer_8hidden on task 0.1q
Using a scheduler
Training FC_16_GRU_1layer_8hidden on task 0.1q, saving to D:\Projects\mental-rotations\models\FC_16_GRU_1layer_8hidden_0.1q_model_gradual_constant; copy the below for logs
tensorboard --logdir=D:\Projects\mental-rotations\logs\FC_16_GRU_1layer_8hidden_0.1q_model_gradual_constant
Training split 1
Directory does not exist: D:\Projects\mental-rotations\logs\FC_16_GRU_1layer_8hidden_0.1q_model_grad

In [8]:
trainer.model.flatten

Custom_CNN(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (rnn_model): FC_RNN(
    (rnn): GRU(16, 8, batch_first=True)
    (fc_in): Sequential(
      (0): Linear(in_features=64, out_features=16, bias=True)
      (1): ReLU()
    )
    (fc_out): Linear(in_features=8, out_features=3, bias=True)
  )
)

In [5]:
task = "0.1q"
network_best_split = {
    "FC_16_GRU_1layer_8hidden": 3,
    # "FC_32_GRU_1layer_8hidden": 3,
    # "FC_16_GRU_1layer_16hidden": 1,
    # "FC_32_GRU_1layer_16hidden": 5,
    # "FC_16_GRU_1layer_32hidden": 5,
    # "FC_32_GRU_1layer_32hidden": 3,
}
for network_name, split in network_best_split.items():
    epochs = 1000
    config = train.build_config(network_name, task, 0, 1, special_names="constant")
    config['optimizer_specs']['optimizer_params']["lr"] = 0.01
    trainer = t.Trainer(config)
    trainer.load_checkpoint(config['check_path'] + f"\\split_{split}\\checkpoint99.pth")
    # trainer.scheduler = None
    config['training_config'].pop('data_save_path')
    data = g.gen_training_data(config['training_config'])
    import utils as u
    reload(u)
    dataloaders = u.get_dataloaders(data, config['training_config']['mini_batch_size'], 1)
    print("Continue training the best model")
    sub_log_path = config['log_path'] + "\\continue_training"
    sub_check_path = config['check_path'] + "\\continue_training"
    force_remove_dir(sub_log_path)
    force_remove_dir(sub_check_path)
    os.makedirs(sub_log_path, exist_ok=True)
    os.makedirs(sub_check_path, exist_ok=True)
    trainer.train(
        dataloaders[0][0],
        dataloaders[0][1],
        epochs=epochs,
        save_path = config['save_path'] + "\\continue_training.pth",
        check_path=sub_check_path,
        log_path = sub_log_path,
    )
    trainer.save_model(config['save_path'] + "\\continue_training.pth", full = 1)
    trainer.load_best_model()
    trainer.save_model(config['save_path'] + "\\best_model.pth", full = 1)

Path already exists for FC_16_GRU_1layer_8hidden in task 0.1q. Assume the model is already trained.
Path already exists for FC_16_GRU_1layer_8hidden in task 0.1q. Assume the model is already trained.
Path already exists for FC_16_GRU_1layer_8hidden in task 0.1q. Assume the model is already trained.
Using a scheduler
Continue training the best model
Successfully removed directory: D:\Projects\mental-rotations\logs\FC_16_GRU_1layer_8hidden_0.1q_model_gradual_constant\continue_training
Successfully removed directory: D:\Projects\mental-rotations\models\FC_16_GRU_1layer_8hidden_model_checkpoints_gradual_constant\continue_training


# CNN_RNN work starts here

## Start by building a CNN from Config

In [2]:
CNN_config = train.build_config_cnn("CNN_FC_16_GRU_1layer_8hidden", "1.1", 0, 1, special_names="constant")

Path already exists for FC_16_GRU_1layer_8hidden in task 0.1q. Assume the model is already trained.
Path already exists for FC_16_GRU_1layer_8hidden in task 0.1q. Assume the model is already trained.
Path already exists for FC_16_GRU_1layer_8hidden in task 0.1q. Assume the model is already trained.
Path already exists for CNN_FC_16_GRU_1layer_8hidden in task 1.1. Assume the model is already trained.
Path already exists for CNN_FC_16_GRU_1layer_8hidden in task 1.1. Assume the model is already trained.
Path already exists for CNN_FC_16_GRU_1layer_8hidden in task 1.1. Assume the model is already trained.


In [7]:
CNN_trainer = t.Trainer(CNN_config)

Using a scheduler


In [9]:
CNN_trainer.model.rnn_model.load_state_dict(trainer.model.rnn.state_dict())


RuntimeError: Error(s) in loading state_dict for CustomRNN:
	Missing key(s) in state_dict: "rnn.weight_ih_l0", "rnn.weight_hh_l0", "rnn.bias_ih_l0", "rnn.bias_hh_l0", "fc.weight", "fc.bias". 
	Unexpected key(s) in state_dict: "weight_ih_l0", "weight_hh_l0", "bias_ih_l0", "bias_hh_l0". 

In [10]:
cnn_rnn_dict = CNN_trainer.model.rnn_model.state_dict()

In [14]:
import torch
import torchvision.models as models
from torchvision.models import ConvNeXt_Tiny_Weights

In [15]:
weights = ConvNeXt_Tiny_Weights.IMAGENET1K_V1
model = models.convnext_tiny(weights)

C:\Users\timmy\anaconda3\envs\mental-rotations\lib\site-packages\torchvision\models\_utils.py:135: UserWarning: Using 'weights' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to C:\Users\timmy/.cache\torch\hub\checkpoints\convnext_tiny-983f1562.pth
100%|██████████| 109M/109M [00:04<00:00, 28.1MB/s] 


In [38]:
preprocess = weights.DEFAULT.transforms()
image = torch.rand(12, 3, 64, 64)
image_p = preprocess(image)

In [76]:
import torch.nn as nn
class ExperimentalModel(nn.Module): 
    def __init__(self, convnet, rnn_type = None): 
        super(ExperimentalModel, self).__init__() 
        if type(convnet) == str: 
            try: 
                weights = __import__("torchvision.models", fromlist = [convnet + "_Weights"])
                model = __import__("torchvision.models", fromlist = [convnet.lower()])
                self.conv = model(weights)
            except Exception as e: 
                raise NotImplementedError(f"{e}, wait for the full version!")
        elif isinstance(convnet, nn.Module): 
            self.conv = convnet
         
        self.conv_output_size = self.conv(torch.randn((1, 3, 64, 64))).shape[-3]
        
        # pool the last layer
        self.postprocess = nn.Sequential(
            nn.AdaptiveAvgPool2d(output_size=1),
            nn.Flatten(),
            nn.LayerNorm(self.conv_output_size), 
        )
        self.total_period = 100
        self.action_period = 50
        self.prep_period = self.total_period - self.action_period
        
        self.linear1_size = 64
        self.linear2_size = 32
        self.rnn_input_size = 16
        self.processing = nn.Sequential(
            nn.Linear(self.conv_output_size, self.linear1_size), 
            nn.ReLU(), 
            nn.Linear(self.linear1_size, self.linear2_size), 
            nn.ReLU(), 
            nn.Linear(self.linear2_size, self.rnn_input_size), 
            nn.ReLU()
        )
        if rnn_type is not None: 
            self.rnn = rnn_type
            self.hidden_size = self.rnn.hidden_size
        else: 
            self.rnn = nn.GRU(self.rnn_input_size, 8, 1, True) # BATCH FIRST
            self.hidden_size = 8
            
        self.output = nn.Linear(self.hidden_size, 3)
            
    def forward(self, x):
        """ Implements a simple forward loop.
        
        Args: x (batch_size, 3, dim1, dim2)
        """
        x = self.conv(x)
        x = self.postprocess(x)
        x = self.processing(x)
        batch_size, processed_size = x.shape
        x = torch.cat((x.unsqueeze(1).repeat(1, self.prep_period, 1), torch.zeros((batch_size, self.action_period, 
                                                                                   processed_size)
                                                                                  )), dim = 1)
        
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device).float()
        x, _ = self.rnn(x, h0)       
        x = self.output(x)
        print(x.shape)


dummy_input = torch.rand(5, 3, 128, 128)
experiment = ExperimentalModel(model.features) 
experiment(dummy_input)

torch.Size([5, 100, 3])


In [54]:
dummy_output = model.features(dummy_input)

In [26]:
nn.GroupNorm(num_groups = 1, num_channels = 768)

GroupNorm(1, 768, eps=1e-05, affine=True)

In [47]:
import torchvision
avgpool = nn.AdaptiveAvgPool2d(output_size=1)
layernorm = torchvision.models.convnext.LayerNorm2d(768)

In [55]:
dummy_output_avg = avgpool(dummy_output)

In [52]:
print(model)

ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

In [60]:
isinstance(model, nn.Module)

True

Sequential(
  (0): Conv2dNormActivation(
    (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
  )
  (1): Sequential(
    (0): CNBlock(
      (block): Sequential(
        (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
        (1): Permute()
        (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
        (3): Linear(in_features=96, out_features=384, bias=True)
        (4): GELU(approximate='none')
        (5): Linear(in_features=384, out_features=96, bias=True)
        (6): Permute()
      )
      (stochastic_depth): StochasticDepth(p=0.0, mode=row)
    )
    (1): CNBlock(
      (block): Sequential(
        (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
        (1): Permute()
        (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
        (3): Linear(in_features=96, out_features=384, bias=True)
        (4): GELU(approximate='none')